### Estudo de Caso 2: O Problema da Mistura de Petróleo (Médio Porte - 12 Variáveis)
* **Referência:** Goldbarg & Luna (2005), Capítulo 2, Seção 2.2.2, Exemplo 5.
* **Objetivo:** Determinar a quantidade de quatro tipos de petróleo bruto a serem misturados para produzir três tipos de gasolina (Amarela, Azul e Superazul), maximizando o lucro da refinaria.

#### Disponibilidade e Custo do Petróleo Bruto
| Tipo de Petróleo | Quantidade Máxima (Barril/dia) | Custo (R$/Barril) |
| :---: | :---: | :---: |
| **1** | 3.500 | 19 |
| **2** | 2.200 | 24 |
| **3** | 4.200 | 20 |
| **4** | 1.800 | 27 |

#### Especificações e Preço de Venda das Gasolinas
| Tipo de Gasolina | Preço de Venda (R$/Barril) | Especificação de Mistura |
| :---: | :---: | :---: |
| **Superazul** | 35 | Não mais que 30% de 1<br>Não menos que 40% de 2<br>Não mais que 50% de 3 |
| **Azul** | 28 | Não mais que 30% de 1<br>Não menos que 10% de 2 |
| **Amarela** | 22 | Não mais que 70% de 1 |

In [2]:
##Import
import gurobipy as gp
from gurobipy import GRB

In [3]:
# definindo o modelo
try:
    m = gp.Model("Mistura_Petroleo_Medio_Porte")
except gp.GurobiError as e:
    print(f"Erro no Gurobi: {e}")   

Restricted license - for non-production use only - expires 2027-11-29


In [4]:
## Definindo variaveis de descisão:

# Gasolina Amarela (A)
xA1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xA1")
xA2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xA2")
xA3 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xA3")
xA4 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xA4")

# Gasolina Azul (Z)
xZ1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xZ1")
xZ2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xZ2")
xZ3 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xZ3")
xZ4 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xZ4")

# Gasolina Superazul (S)
xS1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xS1")
xS2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xS2")
xS3 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xS3")
xS4 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="xS4")

# Variáveis auxiliares para os totais de cada gasolina
total_A = xA1 + xA2 + xA3 + xA4
total_Z = xZ1 + xZ2 + xZ3 + xZ4
total_S = xS1 + xS2 + xS3 + xS4

In [7]:
## Definindo a função Objetivo = Maximizar Lucro = Receita - Custo
receita = 22 * total_A + 28 * total_Z + 35 * total_S
custo = 19 * (xA1 + xZ1 + xS1) + 24 * (xA2 + xZ2 + xS2) + 20 * (xA3 + xZ3 + xS3) + 27 * (xA4 + xZ4 + xS4)
m.setObjective(receita - custo, GRB.MAXIMIZE)

In [8]:
## Definindo as restrições de disponibilidade de petroleo
m.addConstr(xA1 + xZ1 + xS1 <= 3500, "Disp_Petroleo1")
m.addConstr(xA2 + xZ2 + xS2 <= 2200, "Disp_Petroleo2")
m.addConstr(xA3 + xZ3 + xS3 <= 4200, "Disp_Petroleo3")
m.addConstr(xA4 + xZ4 + xS4 <= 1800, "Disp_Petroleo4")

<gurobi.Constr *Awaiting Model Update*>

In [9]:
## Definição as restrições de especificação da mistura
# gasolina superazul
m.addConstr(xS1 <= 0.3 * total_S, "S_Max30_Pet1")
m.addConstr(xS2 >= 0.4 * total_S, "S_Min40_Pet2")
m.addConstr(xS3 <= 0.5 * total_S, "S_Max50_Pet3")

# gasolina azul
m.addConstr(xZ1 <= 0.3 * total_Z, "Z_Max30_Pet1")
m.addConstr(xZ2 >= 0.1 * total_Z, "Z_Min10_Pet2")

# gasolina amarela
m.addConstr(xA1 <= 0.7 * total_A, "A_Max70_Pet1")

# Não negatividade: lb=0.0

<gurobi.Constr *Awaiting Model Update*>

In [10]:
# otimização
m.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Manjaro Linux")

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 12 columns and 36 nonzeros (Max)
Model fingerprint: 0x18b88e9e
Model has 12 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-01, 1e+00]
  Objective range  [1e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+03, 4e+03]

Presolve removed 2 rows and 1 columns
Presolve time: 0.01s
Presolved: 8 rows, 11 columns, 37 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0194874e+05   2.174010e+03   0.000000e+00      0s
       6    9.9777778e+04   0.000000e+00   0.000000e+00      0s

Solved in 6 iterations and 0.01 seconds (0.00 work units)
Optimal objective  9.977777778e+04


In [14]:
if m.status == GRB.OPTIMAL:
        print("\n" + "="*50)
        print("SOLUÇÃO ÓTIMA DA MISTURA DE PETRÓLEO")
        print("="*50)
        print(f"Lucro Máximo Diário: R$ {m.ObjVal:,.2f}\n")
        
        print("Produção de Gasolina Amarela:")
        print(f"  Petróleo 1: {xA1.X:,.2f} | Petróleo 2: {xA2.X:,.2f} | Petróleo 3: {xA3.X:,.2f} | Petróleo 4: {xA4.X:,.2f}")
        
        print("\nProdução de Gasolina Azul:")
        print(f"  Petróleo 1: {xZ1.X:,.2f} | Petróleo 2: {xZ2.X:,.2f} | Petróleo 3: {xZ3.X:,.2f} | Petróleo 4: {xZ4.X:,.2f}")
        
        print("\nProdução de Gasolina Superazul:")
        print(f"  Petróleo 1: {xS1.X:,.2f} | Petróleo 2: {xS2.X:,.2f} | Petróleo 3: {xS3.X:,.2f} | Petróleo 4: {xS4.X:,.2f}")
else:
        print("O modelo não encontrou solução ótima.")


SOLUÇÃO ÓTIMA DA MISTURA DE PETRÓLEO
Lucro Máximo Diário: R$ 99,777.78

Produção de Gasolina Amarela:
  Petróleo 1: 0.00 | Petróleo 2: 0.00 | Petróleo 3: 0.00 | Petróleo 4: 0.00

Produção de Gasolina Azul:
  Petróleo 1: 2,466.67 | Petróleo 2: 822.22 | Petróleo 3: 3,166.67 | Petróleo 4: 1,766.67

Produção de Gasolina Superazul:
  Petróleo 1: 1,033.33 | Petróleo 2: 1,377.78 | Petróleo 3: 1,033.33 | Petróleo 4: 0.00


In [13]:
# Lista todas as variáveis e restrições que estão carregadas no Gurobi 
print("Variáveis:", [v.varName for v in m.getVars()])
print("Restrições:", [c.constrName for c in m.getConstrs()])

Variáveis: ['xA1', 'xA2', 'xA3', 'xA4', 'xZ1', 'xZ2', 'xZ3', 'xZ4', 'xS1', 'xS2', 'xS3', 'xS4']
Restrições: ['Disp_Petroleo1', 'Disp_Petroleo2', 'Disp_Petroleo3', 'Disp_Petroleo4', 'S_Max30_Pet1', 'S_Min40_Pet2', 'S_Max50_Pet3', 'Z_Max30_Pet1', 'Z_Min10_Pet2', 'A_Max70_Pet1']


* **Conclusão:**
  - O Gurobi otimiza estritamente o que foi modelado. Decidiu descontinuar um produto inteiro (a gasolina amarela) e esgotar seletivamente o estoque dos petróleos mais baratos para alavancar a margem de lucro dos produtos (As gasolinas Superazul e Azul possuem preços de venda muito superiores R$ 35,00 e R$ 28,00)